# 09 - Cross-Source Reconciliation

This notebook consolidates core consistency checks across evidence artifacts and live API context.

**Objective:** detect ambiguity/inconsistency before investor-facing conclusions are circulated.


## Checks implemented

1. TVL baseline vs latest API snapshot (`check_tvl_reconciliation`).
2. Observed fee rate vs claimed 0.66% (`check_fee_rate_consistency`).
3. Observed staker share vs claimed 20% (`check_staker_share`).

The output is a clear PASS/PARTIAL table with tolerance settings shown explicitly.


In [ ]:
from pathlib import Path
import sys
import json
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")

cwd = Path.cwd().resolve()
root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "notebooks" / "notebook_utils.py").exists():
        root = candidate
        break
if root is None:
    raise RuntimeError("Could not locate repository root containing notebooks/notebook_utils.py")

if root.as_posix() not in sys.path:
    sys.path.insert(0, root.as_posix())

from notebooks import notebook_utils as nbx
ROOT = nbx.setup_notebook_context()
print(f"Project root: {ROOT}")
print(f"Notebook run started (UTC): {nbx.utc_now_iso()}")


In [ ]:
from src.reconciliation.checks import (
    check_tvl_reconciliation,
    check_fee_rate_consistency,
    check_staker_share,
)

EVIDENCE_DIR = nbx.latest_evidence_dir()
if EVIDENCE_DIR is None:
    raise FileNotFoundError("No evidence directory found.")

kpi = nbx.load_json(EVIDENCE_DIR / "kpi_summary.json")
payout_ratios = pd.read_csv(EVIDENCE_DIR / "forum_payout_ratios.csv")

nbx.refresh_defillama_snapshots()
lazy_tvl_path = nbx.latest_defillama_snapshot("lazy-summer-protocol", "tvl")
fees_path = nbx.latest_defillama_snapshot("lazy-summer-protocol", "fees")

lazy_tvl = nbx.protocol_tvl_df(nbx.load_json(lazy_tvl_path))
fees_df = nbx.fee_series_df(nbx.load_json(fees_path))


In [ ]:
baseline_tvl = float(kpi.get("lazy_latest_tvl_usd") or 0.0)
latest_api_tvl = float(lazy_tvl["totalLiquidityUSD"].iloc[-1]) if not lazy_tvl.empty else np.nan

fee_summary = nbx.summarize_fee_productivity(lazy_tvl, fees_df, fee_window_days=90)
observed_fee_rate = float(fee_summary.get("implied_fee_rate") or 0.0)

observed_staker_share = float(payout_ratios["staker_share_pct"].mean() / 100.0)

checks = {
    "TVL_reconciliation": check_tvl_reconciliation(
        onchain_tvl=baseline_tvl,
        defillama_tvl=latest_api_tvl,
        tolerance=0.05,
    ),
    "Fee_rate_consistency": check_fee_rate_consistency(
        computed_rate=observed_fee_rate,
        claimed_rate=0.0066,
        tolerance=0.002,
    ),
    "Staker_share_consistency": check_staker_share(
        computed_share=observed_staker_share,
        claimed_share=0.20,
        tolerance=0.03,
    ),
}

rows = []
for name, payload in checks.items():
    rows.append(
        {
            "check": name,
            "passes": bool(payload.get("passes")),
            "details": json.dumps(payload),
        }
    )

checks_df = pd.DataFrame(rows)
display(checks_df)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2), constrained_layout=True)
plot_df = checks_df.copy()
plot_df["pass_int"] = plot_df["passes"].astype(int)
colors = ["#2a9d8f" if v else "#d62828" for v in plot_df["passes"]]
ax.barh(plot_df["check"], plot_df["pass_int"], color=colors)
ax.set_xlim(0, 1)
ax.set_xlabel("Pass (1=yes)")
ax.set_title("Core Reconciliation Checks")
plt.show()


In [ ]:
pass_count = int(checks_df["passes"].sum())
total_count = len(checks_df)
status = "PASS" if pass_count == total_count else "PARTIAL"

summary_md = f"""
**Overall reconciliation status:** `{status}`  
- Passed checks: `{pass_count}/{total_count}`
- Baseline TVL (evidence): `{nbx.fmt_usd(baseline_tvl)}`
- Latest API TVL: `{nbx.fmt_usd(latest_api_tvl)}`
- Observed 90d fee rate: `{nbx.fmt_pct(observed_fee_rate)}`
- Observed staker share (forum-claimed): `{nbx.fmt_pct(observed_staker_share)}`

Interpretation policy: if any check fails, treat downstream investor conclusions as conditional until discrepancy is resolved.
"""
display(Markdown(summary_md))
